<a href="https://colab.research.google.com/github/visionbyangelic/Brain-Aging/blob/main/02_brainage_model_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/ANR_BrainAge")

# 1. Load the RELAXED cohort (this is the actual Fix 1: not the Strict 248-file)
relaxed_path = ROOT / "manifests" / "openbhb" / "openBHB_approved_RELAXED_with_FreeSurfer.csv"
relaxed = pd.read_csv(relaxed_path)
relaxed["participant_id"] = relaxed["participant_id"].astype(str).str.strip()
print(f"Relaxed OpenBHB manifest: {len(relaxed):,} participants")

# 2. Load the FreeSurfer Desikan regional feature table, downloading if it's missing
fs_path = ROOT / "data" / "openbhb" / "openBHB_Desikan_FreeSurfer_features.csv"

if fs_path.exists():
    fs = pd.read_csv(fs_path)
    print(f"Loaded existing FreeSurfer feature table: {fs.shape}")
else:
    print("Not found on Drive, downloading from source (this can take a few minutes)...")
    fs_url = (
        "https://huggingface.co/datasets/benoit-dufumier/openBHB/"
        "resolve/main/train/derivatives/freesurfer_roi/desikan_roi_features.csv"
    )
    fs = pd.read_csv(fs_url)
    fs_path.parent.mkdir(parents=True, exist_ok=True)
    fs.to_csv(fs_path, index=False)
    print(f"Downloaded and saved: {fs.shape} -> {fs_path}")

fs["participant_id"] = fs["participant_id"].astype(str).str.strip()

# 3. Link the RELAXED cohort to FreeSurfer features (replaces the buggy Strict-cohort link)
relaxed_ids = set(relaxed["participant_id"])
fs_ids = set(fs["participant_id"])
matched_ids = relaxed_ids & fs_ids

print(f"\nRelaxed cohort participants: {len(relaxed_ids):,}")
print(f"FreeSurfer table participants: {len(fs_ids):,}")
print(f"Matched (new modeling cohort): {len(matched_ids):,}")
print(f"Coverage: {len(matched_ids) / len(relaxed_ids) * 100:.2f}%")

# 4. Build the aligned cohort + feature matrix, sorted and order-checked
matched_relaxed = relaxed[relaxed["participant_id"].isin(matched_ids)].sort_values("participant_id").reset_index(drop=True)
matched_fs = fs[fs["participant_id"].isin(matched_ids)].drop_duplicates(subset="participant_id").sort_values("participant_id").reset_index(drop=True)

assert (matched_relaxed["participant_id"].values == matched_fs["participant_id"].values).all(), \
    "Participant ordering mismatch between manifest and feature table"

# 5. Save as new files (keeps the old 221-participant run untouched for comparison)
out_manifest_path = ROOT / "manifests" / "openbhb" / "openBHB_phase0_FreeSurfer_linked_cohort_RELAXED.csv"
out_features_path = ROOT / "data" / "openbhb" / "openBHB_phase0_modeling_feature_matrix_RELAXED.csv"

matched_relaxed.to_csv(out_manifest_path, index=False)
matched_fs.to_csv(out_features_path, index=False)

print(f"\nSaved: {out_manifest_path}")
print(f"Saved: {out_features_path}")
print(f"\nCorrected N = {len(matched_relaxed):,}  (previous buggy run: N = 221)")

# Quick sanity check: does the larger relaxed pool pull in a wider age range?
if "age" in matched_relaxed.columns:
    print("\nAge distribution of the corrected cohort:")
    print(matched_relaxed["age"].describe())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Relaxed OpenBHB manifest: 3,240 participants
Loaded existing FreeSurfer feature table: (3227, 478)

Relaxed cohort participants: 3,240
FreeSurfer table participants: 3,227
Matched (new modeling cohort): 2,598
Coverage: 80.19%

Saved: /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_phase0_FreeSurfer_linked_cohort_RELAXED.csv
Saved: /content/drive/MyDrive/ANR_BrainAge/data/openbhb/openBHB_phase0_modeling_feature_matrix_RELAXED.csv

Corrected N = 2,598  (previous buggy run: N = 221)

Age distribution of the corrected cohort:
count    2598.000000
mean       20.718025
std         8.930779
min         6.000000
25%        18.000000
50%        21.000000
75%        23.000000
max        83.000000
Name: age, dtype: float64


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import joblib
import json

ROOT = Path("/content/drive/MyDrive/ANR_BrainAge")

# ============================================================
# 1. Load corrected cohort + locked Option B parity mapping
# ============================================================
manifest = pd.read_csv(ROOT / "manifests" / "openbhb" / "openBHB_phase0_FreeSurfer_linked_cohort_RELAXED.csv")
fs = pd.read_csv(ROOT / "data" / "openbhb" / "openBHB_phase0_modeling_feature_matrix_RELAXED.csv")

parity_path = ROOT / "manifests" / "feature_parity" / "openBHB_OASIS3_phase0_regional_measurement_compatibility.csv"
parity = pd.read_csv(parity_path)

openbhb_col = "openbhb_feature" if "openbhb_feature" in parity.columns else parity.columns[0]
option_b_features = parity[openbhb_col].dropna().unique().tolist()

missing = [f for f in option_b_features if f not in fs.columns]
if missing:
    print(f"WARNING: {len(missing)} Option B features not found in the FreeSurfer table, dropping them:")
    print(missing[:10])
option_b_features = [f for f in option_b_features if f in fs.columns]
print(f"Option B features usable: {len(option_b_features)}")

# ============================================================
# 2. Build the modeling table (participant-level)
# ============================================================
manifest["participant_id"] = manifest["participant_id"].astype(str).str.strip()
fs["participant_id"] = fs["participant_id"].astype(str).str.strip()

model_df = fs[["participant_id"] + option_b_features].merge(
    manifest[["participant_id", "age"]], on="participant_id", how="inner"
)
model_df = model_df.dropna(subset=["age"] + option_b_features).reset_index(drop=True)

print(f"\nFinal modeling table: {model_df.shape[0]} participants, {len(option_b_features)} features")
print(f"Age: min={model_df['age'].min():.1f}  mean={model_df['age'].mean():.1f}  max={model_df['age'].max():.1f}")

# ============================================================
# 3. Participant-level 70/15/15 split
# ============================================================
train_df, temp_df = train_test_split(model_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
print(f"\nSplit -> train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

X_train, y_train = train_df[option_b_features], train_df["age"]
X_val, y_val = val_df[option_b_features], val_df["age"]
X_test, y_test = test_df[option_b_features], test_df["age"]

# ============================================================
# 4. Model ladder: mean -> Ridge -> Elastic Net -> SVR -> Random Forest
# ============================================================
def make_pipeline(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])

candidates = {
    "Mean_Baseline": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
    "SVR": SVR(kernel="rbf", C=1.0, epsilon=0.5),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
}

results, fitted = [], {}
print("\nTraining model ladder (SVR/RF may take a minute on this larger N)...")
for name, est in candidates.items():
    pipe = make_pipeline(est)
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    mae = mean_absolute_error(y_val, val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    r2 = r2_score(y_val, val_pred)
    r = 0.0 if name == "Mean_Baseline" else stats.pearsonr(y_val, val_pred)[0]
    results.append({"model": name, "val_MAE": mae, "val_RMSE": rmse, "val_R2": r2, "val_pearson_r": r})
    fitted[name] = pipe
    print(f"  {name:15s} val MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}  r={r:.3f}")

results_df = pd.DataFrame(results).sort_values("val_MAE")
non_baseline = results_df[results_df["model"] != "Mean_Baseline"]
best_name = non_baseline.iloc[0]["model"]
best_pipe = fitted[best_name]
print(f"\nSelected (by validation MAE, excluding baseline): {best_name}")

# ============================================================
# 5. Bias calibration, fit on TRAIN+VAL only
# ============================================================
dev_df = pd.concat([train_df, val_df])
dev_pred = best_pipe.predict(dev_df[option_b_features])
bag_dev = dev_pred - dev_df["age"]
slope, intercept, _, _, _ = stats.linregress(dev_df["age"], bag_dev)
print(f"\nBias calibration (N={len(dev_df)}): E[BAG_raw | age] = {intercept:.4f} + ({slope:.4f}) * age")

# ============================================================
# 6. Evaluate ONCE on held-out test
# ============================================================
test_pred_raw = best_pipe.predict(X_test)
bag_raw_test = test_pred_raw - y_test
test_pred_calibrated = test_pred_raw - (intercept + slope * y_test)
bag_calibrated_test = test_pred_calibrated - y_test

def report(label, pred):
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    r = stats.pearsonr(y_test, pred)[0]
    print(f"  {label}: MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}  r={r:.3f}")
    return mae, rmse, r2, r

print(f"\nHeld-out test (N={len(test_df)}):")
raw_m = report("Raw", test_pred_raw)
cal_m = report("Calibrated", test_pred_calibrated)

# ============================================================
# 7. Freeze artifacts (RELAXED suffix, originals untouched)
# ============================================================
models_dir, manifests_dir = ROOT / "models", ROOT / "manifests" / "openbhb"
models_dir.mkdir(parents=True, exist_ok=True)
manifests_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_pipe, models_dir / "frozen_openbhb_normative_pipeline_RELAXED.joblib")
with open(models_dir / "frozen_openbhb_model_config_RELAXED.json", "w") as f:
    json.dump({
        "selected_model": best_name,
        "feature_list": option_b_features,
        "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
        "bias_calibration": {"intercept": intercept, "slope": slope},
        "val_ladder": results_df.to_dict(orient="records"),
        "test_raw": dict(zip(["MAE","RMSE","R2","pearson_r"], raw_m)),
        "test_calibrated": dict(zip(["MAE","RMSE","R2","pearson_r"], cal_m)),
    }, f, indent=2, default=float)

test_out = test_df[["participant_id", "age"]].copy()
test_out["predicted_age_raw"] = test_pred_raw
test_out["predicted_age_calibrated"] = test_pred_calibrated
test_out["BAG_raw"] = bag_raw_test
test_out["BAG_calibrated"] = bag_calibrated_test
test_out.to_csv(manifests_dir / "openBHB_test_predictions_RELAXED.csv", index=False)

print("\nFrozen pipeline, config, and test predictions saved with _RELAXED suffix.")

Option B features usable: 136

Final modeling table: 2598 participants, 136 features
Age: min=6.0  mean=20.7  max=83.0

Split -> train: 1818  val: 390  test: 390

Training model ladder (SVR/RF may take a minute on this larger N)...
  Mean_Baseline   val MAE=5.176  RMSE=8.583  R2=-0.004  r=0.000
  Ridge           val MAE=4.106  RMSE=6.079  R2=0.496  r=0.715
  ElasticNet      val MAE=3.885  RMSE=6.060  R2=0.499  r=0.711
  SVR             val MAE=3.454  RMSE=6.810  R2=0.368  r=0.642
  RandomForest    val MAE=3.467  RMSE=5.776  R2=0.545  r=0.745

Selected (by validation MAE, excluding baseline): SVR

Bias calibration (N=2208): E[BAG_raw | age] = 13.2668 + (-0.7107) * age

Held-out test (N=390):
  Raw: MAE=3.778  RMSE=7.492  R2=0.297  r=0.633
  Calibrated: MAE=2.168  RMSE=2.711  R2=0.908  r=0.954

Frozen pipeline, config, and test predictions saved with _RELAXED suffix.


In [ ]:
# Sanity check: is the calibrated accuracy even across ages, or driven by a few tail points?
test_df_check = test_df.copy()
test_df_check["predicted_calibrated"] = test_pred_calibrated
test_df_check["BAG_calibrated"] = bag_calibrated_test
test_df_check["age_decade"] = (test_df_check["age"] // 10 * 10).astype(int)

decade_summary = test_df_check.groupby("age_decade").agg(
    n=("age", "count"),
    mean_age=("age", "mean"),
    mean_BAG_calibrated=("BAG_calibrated", "mean"),
    MAE_calibrated=("BAG_calibrated", lambda x: x.abs().mean()),
).reset_index()

print(decade_summary.to_string(index=False))

 age_decade   n  mean_age  mean_BAG_calibrated  MAE_calibrated
          0  18  8.817946            -3.131468        3.436734
         10 145 16.314477            -0.316147        2.435986
         20 197 22.887970             1.161140        1.641696
         30  16 33.330267            -0.404379        1.658736
         40   4 44.000000            -3.075244        3.075244
         50   3 53.333333            -4.090005        4.090005
         60   6 64.500000            -7.437178        7.437178
         70   1 74.000000           -11.297192       11.297192


In [ ]:
# ============================================================
# Re-run Experiment 3: frozen RELAXED pipeline -> OASIS-3 HC
# ============================================================
oasis_feat_path = ROOT / "manifests" / "oasis3" / "OASIS3_feature_matrix.csv"
df_oasis = pd.read_csv(oasis_feat_path)

parity_path = ROOT / "manifests" / "feature_parity" / "openBHB_OASIS3_phase0_regional_measurement_compatibility.csv"
parity = pd.read_csv(parity_path)
openbhb_col = "openbhb_feature" if "openbhb_feature" in parity.columns else parity.columns[0]
oasis_col = "oasis3_feature" if "oasis3_feature" in parity.columns else parity.columns[1]

oasis_to_openbhb = dict(zip(parity[oasis_col], parity[openbhb_col]))
oasis_to_openbhb = {k: v for k, v in oasis_to_openbhb.items() if v in option_b_features}

missing_oasis_cols = [c for c in oasis_to_openbhb if c not in df_oasis.columns]
for c in missing_oasis_cols:
    oasis_to_openbhb.pop(c)
if missing_oasis_cols:
    print(f"WARNING: {len(missing_oasis_cols)} mapped OASIS-3 columns not found, dropped.")

age_col_oasis = next((c for c in df_oasis.columns if c.lower() in ["age", "ageatentry"] or "age at" in c.lower()), None)
assert age_col_oasis is not None, "Could not find an age column in OASIS3_feature_matrix.csv"

oasis_model_df = df_oasis[list(oasis_to_openbhb.keys()) + [age_col_oasis]].copy()
oasis_model_df = oasis_model_df.rename(columns=oasis_to_openbhb).rename(columns={age_col_oasis: "age"})
oasis_model_df = oasis_model_df.dropna(subset=["age"])

still_missing = [f for f in option_b_features if f not in oasis_model_df.columns]
if still_missing:
    print(f"NOTE: {len(still_missing)} training features have no OASIS-3 counterpart, imputer will fill these.")
    for f in still_missing:
        oasis_model_df[f] = np.nan

X_oasis = oasis_model_df[option_b_features]
y_oasis = oasis_model_df["age"]

oasis_pred_raw = best_pipe.predict(X_oasis)
oasis_pred_calibrated = oasis_pred_raw - (intercept + slope * y_oasis)

def report_ext(label, pred, y_true):
    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)
    r = stats.pearsonr(y_true, pred)[0]
    print(f"  {label}: MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}  r={r:.3f}")

print(f"\nOASIS-3 HC external evaluation (N={len(oasis_model_df)}), model = {best_name}:")
report_ext("Raw", oasis_pred_raw, y_oasis)
report_ext("Calibrated", oasis_pred_calibrated, y_oasis)

print(f"\nFor comparison, the old buggy N=221 pipeline scored calibrated external MAE = 30.432y (SVR)")


OASIS-3 HC external evaluation (N=1049), model = SVR:
  Raw: MAE=45.391  RMSE=46.326  R2=-24.691  r=-0.015
  Calibrated: MAE=10.466  RMSE=10.888  R2=-0.419  r=0.978

For comparison, the old buggy N=221 pipeline scored calibrated external MAE = 30.432y (SVR)


# ANR Lab Brain-Age Build 1: Fix Log, OpenBHB Cohort Correction

**Date:** August 2026
**Scope:** Stage 0 feature linkage bug affecting Experiment 2 (OpenBHB normative model) and Experiment 3 (cross-dataset external evaluation on OASIS-3)

---

## The problem

`Feature_Compatibility_Assessment.ipynb` explicitly locked the OpenBHB cohort route as **Relaxed Tim Trio** (N = 3,240) in its "Final Locked Decisions" cell. However, the code in the "Phase 0, Final Eligible Cohort Freeze" step that actually linked participants to the external FreeSurfer Desikan regional feature table loaded a different file:

```
openBHB_approved_manifest.csv
```

That file is the original **Strict** cohort (N = 248), created earlier in `OpenBHB_Filtering.ipynb`, not the Relaxed Tim Trio file (`openBHB_approved_RELAXED_TimTrio.csv` / `openBHB_approved_RELAXED_with_FreeSurfer.csv`, N = 3,240). The written decision and the executed code disagreed, silently.

Because of this, the FreeSurfer linkage matched only 221 of the 248 Strict participants, and every downstream step, Experiment 2's training set, its held-out test set, and the frozen pipeline used in Experiment 3, was built on that undersized, mislabeled cohort.

## Why it mattered

- Experiment 2 trained on only 154 participants with a held-out test of 34, far smaller than the actual data available.
- The Strict cohort's age range was narrower than the full Relaxed pool, limiting how much of OpenBHB's older tail the model ever saw.
- The mismatch between documented decision and executed code was itself a reproducibility problem, on top of any accuracy cost.
- A smaller, narrower training set is a plausible contributor to how badly Experiment 3 collapsed when the frozen model was evaluated against OASIS-3's much older cohort (calibrated external MAE of 30.4 years).

## What we did

1. Identified the correct Relaxed Tim Trio manifest with FreeSurfer availability already flagged (`openBHB_approved_RELAXED_with_FreeSurfer.csv`, N = 3,240).
2. Re-established the canonical path for the external FreeSurfer Desikan feature table (`data/openbhb/openBHB_Desikan_FreeSurfer_features.csv`) and added a fallback that re-downloads it from the original Hugging Face source if it is ever missing from Drive.
3. Re-ran the participant-level linkage directly between the full 3,240-participant Relaxed cohort and the FreeSurfer feature table, bypassing the buggy Strict-cohort intermediate entirely.
4. Rebuilt the 136-feature Option B modeling matrix from the corrected cohort, reusing the feature-parity mapping already locked in Stage 0 (no need to redo that decision).
5. Re-ran the full model ladder (participant-level 70/15/15 split, mean baseline through Random Forest, selection by validation MAE, linear bias calibration fit on train+val only, single held-out test evaluation) and froze new artifacts under a `_RELAXED` suffix, leaving the original N = 221 run untouched for comparison.
6. Re-ran Experiment 3 (frozen pipeline applied to OASIS-3 healthy controls, N = 1,049) using the same corrected pipeline.

## Results

### Corrected cohort

| | Before (buggy) | After (fixed) |
|---|---|---|
| Matched participants | 221 | 2,598 |
| Source file | Strict manifest (248) | Relaxed Tim Trio manifest (3,240) |
| Coverage | 89.1% of 248 | 80.2% of 3,240 |
| Age range | narrower, capped low | 6 to 83 (mean 20.7) |
| Train / val / test | 154 / 33 / 34 | 1,818 / 390 / 390 |

### Experiment 2: OpenBHB normative model, re-run

Validation ladder:

| Model | Val MAE | Val R2 |
|---|---:|---:|
| Mean baseline | 5.176 | -0.004 |
| Ridge | 4.106 | 0.496 |
| Elastic Net | 3.885 | 0.499 |
| **SVR (selected)** | **3.454** | 0.368 |
| Random Forest | 3.467 | **0.545** |

SVR was selected on validation MAE by a narrow margin, but Random Forest explained meaningfully more variance (R2 0.545 vs 0.368). Worth revisiting which model actually carries forward.

Bias calibration (fit on train+val, N = 2,208): `E[BAG_raw | age] = 13.2668 + (-0.7107) x age`

Held-out test (N = 390):

| | MAE | RMSE | R2 | r |
|---|---:|---:|---:|---:|
| Raw | 3.778 | 7.492 | 0.297 | 0.633 |
| Calibrated | 2.168 | 2.711 | 0.908 | 0.954 |

Decade-level breakdown of the calibrated test set:

| Age decade | n | Mean age | Mean BAG (calibrated) | MAE (calibrated) |
|---:|---:|---:|---:|---:|
| 0s | 18 | 8.8 | -3.13 | 3.44 |
| 10s | 145 | 16.3 | -0.32 | 2.44 |
| 20s | 197 | 22.9 | 1.16 | 1.64 |
| 30s | 16 | 33.3 | -0.40 | 1.66 |
| 40s | 4 | 44.0 | -3.08 | 3.08 |
| 50s | 3 | 53.3 | -4.09 | 4.09 |
| 60s | 6 | 64.5 | -7.44 | 7.44 |
| 70s | 1 | 74.0 | -11.30 | 11.30 |

**Note:** the aggregate calibrated R2 of 0.908 is driven almost entirely by the 10s and 20s bins, which hold 342 of the 390 test participants. Every bin above 40 has fewer than 10 people, several have fewer than 5, and error climbs steadily in those bins. Treat the headline R2 as a young-adult accuracy figure, not evidence of even performance across the lifespan.

### Experiment 3: external evaluation on OASIS-3 healthy controls, re-run

| | Before (buggy, N=221) | After (fixed, N=2,598) |
|---|---:|---:|
| Raw external MAE | 47.42y | 45.39y |
| Calibrated external MAE | **30.43y** | **10.47y** |
| Calibrated R2 | not reported | -0.419 |
| Calibrated r | 0.915 | 0.978 |

Fixing the cohort cut the calibrated external MAE from 30.4 years to 10.5 years, roughly a two-thirds reduction. That is a substantial, genuine improvement and the clearest sign the fix was worth doing.

**One nuance to flag:** the calibrated R2 on OASIS-3 is still negative (-0.419) even though MAE dropped sharply and r is very strong (0.978). That combination means the calibrated predictions now track OASIS-3's age ordering closely, but likely sit at a systematic offset from the true ages rather than landing on the identity line. R2 is sensitive to that kind of constant offset in a way MAE and r are not. The bias calibration term was fit entirely on OpenBHB's dev set (mean age ~21), so it is extrapolating to correct predictions for OASIS-3's much older cohort (mean age ~68), and a single linear slope fit on young data may not fully hold at that distance. Worth plotting predicted vs. true age for the OASIS-3 evaluation before treating the 10.5-year MAE as the final word.

## Files produced

- `manifests/openbhb/openBHB_phase0_FreeSurfer_linked_cohort_RELAXED.csv`
- `data/openbhb/openBHB_phase0_modeling_feature_matrix_RELAXED.csv`
- `models/frozen_openbhb_normative_pipeline_RELAXED.joblib`
- `models/frozen_openbhb_model_config_RELAXED.json`
- `manifests/openbhb/openBHB_test_predictions_RELAXED.csv`

All original `_221` artifacts were left in place for comparison.

## Still open

- **Experiment 4 (clinical BAG association)** has not been re-run yet. The OASIS-3 feature matrix used so far only contains CDR = 0 participants; the CDR > 0 clinical subgroup still needs to be pulled in separately and compared using ANCOVA with age as a covariate.
- **Model choice for Experiment 2** deserves a second look given how close SVR and Random Forest were on validation MAE, with Random Forest ahead on R2.
- **Predicted vs. true age plot for OASIS-3** should be generated to understand the negative-R2 result before writing it up as a clean win.